In [ ]:
import os
import numpy as np
import pandas as pd

PROJECT_ROOT = r"C:\path\to\your\supply_chain_project"   # <-- EDIT THIS (same path as before)
OUT = os.path.join(PROJECT_ROOT, "outputs")
os.makedirs(OUT, exist_ok=True)

demand_df = pd.read_csv(f"{OUT}/daily_demand.csv", parse_dates=["date"])
capacity_df = pd.read_csv(f"{OUT}/supplier_capacity.csv", parse_dates=["date"])
config_df = pd.read_csv(f"{OUT}/network_config.csv")
AVG_DAILY_DEMAND = demand_df.groupby("date")["demand_units"].sum().mean()
STARTING_INVENTORY = AVG_DAILY_DEMAND * 3      # 3 days of cover on hand (lean network -> disruptions bite)
SAFETY_STOCK = AVG_DAILY_DEMAND * 2            # 2 days of cover triggers emergency expediting
NORMAL_UNIT_COST = config_df.loc[config_df["type"] == "supplier", "unit_cost"].mean()
EXPEDITE_PREMIUM = 1.40
LOST_SALE_MULTIPLIER = 2.0
# Emergency/expedited capacity is limited (e.g. air freight, spot-market buys) -
# it cannot fully replace lost supplier capacity, which is what allows a real
# stockout (and therefore a measurable service-level hit) during a disruption.
MAX_EXPEDITE_CAPACITY_PER_DAY = AVG_DAILY_DEMAND * 0.20

SIM_DAYS = 90            # total simulation window
DISRUPTION_START = 20     # day disruption begins
DISRUPTION_LENGTH = 14    # how many days it lasts

daily_total_demand = (
    demand_df.groupby("date")["demand_units"].sum().sort_index().reset_index()
)
daily_total_capacity = (
    capacity_df.groupby("date")["capacity_units"].sum().sort_index().reset_index()
)

sim_dates = daily_total_demand["date"].iloc[:SIM_DAYS].reset_index(drop=True)
demand_series = daily_total_demand["demand_units"].iloc[:SIM_DAYS].reset_index(drop=True)
capacity_series = daily_total_capacity["capacity_units"].iloc[:SIM_DAYS].reset_index(drop=True)

# supplier share of capacity (for supplier-failure scenario)
supplier_shares = config_df.dropna(subset=["share_of_supply"]).set_index("node")["share_of_supply"]


def run_simulation(scenario: str):
    """
    scenario: 'baseline' | 'supplier_failure' | 'port_closure'
    Returns a per-day DataFrame with inventory, fulfilled/unfulfilled demand, and daily cost.
    """
    inventory = STARTING_INVENTORY
    rows = []
    port_closure_backlog = 0  # units delayed in transit, released after closure ends

    for i in range(SIM_DAYS):
        day = i
        base_capacity = capacity_series.iloc[i]
        demand_today = demand_series.iloc[i]
        in_disruption = DISRUPTION_START <= day < DISRUPTION_START + DISRUPTION_LENGTH

        effective_capacity = base_capacity
        expedited_units = 0

        if scenario == "supplier_failure" and in_disruption:
            # Supplier_A (largest, 38% share) fails completely
            effective_capacity = base_capacity * (1 - supplier_shares["Supplier_A"])

        elif scenario == "port_closure" and in_disruption:
            # All inbound shipments delayed -> capacity arriving today drops by 55%
            # (goods pile up in transit rather than being lost)
            reduction = base_capacity * 0.55
            effective_capacity = base_capacity - reduction
            port_closure_backlog += reduction

        elif scenario == "port_closure" and day == DISRUPTION_START + DISRUPTION_LENGTH:
            # Port reopens: delayed shipments arrive in a catch-up surge over the next 5 days
            pass  # handled by release schedule below

        # release backlog gradually over 5 days after port reopens
        if scenario == "port_closure" and DISRUPTION_START + DISRUPTION_LENGTH <= day < DISRUPTION_START + DISRUPTION_LENGTH + 5:
            release = port_closure_backlog / 5
            effective_capacity += release

        inventory += effective_capacity

        # if inventory is under safety stock, expedite extra units at a premium
        if inventory < SAFETY_STOCK and scenario != "baseline":
            gap = SAFETY_STOCK - inventory
            expedited_units = min(gap, demand_today, MAX_EXPEDITE_CAPACITY_PER_DAY)
            inventory += expedited_units

        fulfilled = min(inventory, demand_today)
        unfulfilled = max(demand_today - inventory, 0)
        inventory = max(inventory - fulfilled, 0)

        daily_cost = (
            fulfilled * NORMAL_UNIT_COST
            + expedited_units * NORMAL_UNIT_COST * (EXPEDITE_PREMIUM - 1)  # premium only
            + unfulfilled * NORMAL_UNIT_COST * LOST_SALE_MULTIPLIER
        )

        rows.append({
            "day": day, "date": sim_dates.iloc[i], "scenario": scenario,
            "demand": round(demand_today, 1), "capacity_arrived": round(effective_capacity, 1),
            "inventory_end_of_day": round(inventory, 1), "fulfilled": round(fulfilled, 1),
            "unfulfilled": round(unfulfilled, 1),
            "service_level_%": round(100 * fulfilled / demand_today, 2) if demand_today > 0 else 100.0,
            "expedited_units": round(expedited_units, 1),
            "daily_cost": round(daily_cost, 2),
            "in_disruption_window": in_disruption,
        })

    return pd.DataFrame(rows)


baseline = run_simulation("baseline")
supplier_failure = run_simulation("supplier_failure")
port_closure = run_simulation("port_closure")

all_scenarios = pd.concat([baseline, supplier_failure, port_closure], ignore_index=True)
all_scenarios.to_csv(f"{OUT}/simulation_results.csv", index=False)

print("Simulation complete. Scenarios: baseline, supplier_failure, port_closure")
print(f"Disruption window: day {DISRUPTION_START} to {DISRUPTION_START + DISRUPTION_LENGTH - 1}")
print("\nMin service level reached per scenario:")
print(all_scenarios.groupby("scenario")["service_level_%"].min())
print("\nSaved: simulation_results.csv")


Simulation complete. Scenarios: baseline, supplier_failure, port_closure
Disruption window: day 20 to 33

Min service level reached per scenario:
scenario
baseline            100.00
port_closure         57.50
supplier_failure     72.88
Name: service_level_%, dtype: float64

Saved: simulation_results.csv
